# 312 — Graph Tools

Tests every MCP tool against live Neo4j data.

Uses real entity IDs confirmed from the CSV data:
- `LOAN-0002` — LVR=95, high-risk loan
- `BRW-0001` — Patrick Nelson, individual, high risk
- `ACC-0596` — transaction structuring target
- `BRW-0582` — top of layered ownership chain

In [1]:
%run 311_agent_setup.ipynb

/opt/anaconda3/envs/graphrag-finserv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-09 14:37:05,236 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 1, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 1, 'Regulation': 3, 'ReasoningStep': 1, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrieve_

## Tool 1: read-neo4j-cypher (Neo4j MCP)

In [2]:
import json

result = execute_tool('read-neo4j-cypher', {
    'query': """
        MATCH (l:LoanApplication {loan_id: 'LOAN-0002'})-[:SUBMITTED_BY]->(b:Borrower)
        RETURN l.loan_id AS loan_id, l.lvr AS lvr, l.amount AS amount,
               b.borrower_id AS borrower_id, b.name AS name, b.risk_rating AS risk
    """
})
print('read-neo4j-cypher:', json.dumps(result, indent=2, default=str))

2026-03-09 14:37:06,283 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']


read-neo4j-cypher: {
  "rows": [
    {
      "loan_id": "LOAN-0002",
      "lvr": 95,
      "amount": 900000,
      "borrower_id": "BRW-0003",
      "name": "Katarina Wu",
      "risk": "low"
    }
  ]
}


## Tool 2: traverse_compliance_path (FastMCP)

In [3]:
result = execute_tool('traverse_compliance_path', {
    'entity_id': 'LOAN-0002',
    'entity_type': 'LoanApplication',
    'regulation_id': 'APG-223',
})

print('Entity:', result.get('entity', {}))
print('Jurisdiction:', result.get('jurisdiction_id'))
print('\nRegulations found:', list(result.get('regulations', {}).keys()))

for reg_id, reg in result.get('regulations', {}).items():
    for sec_id, sec in reg.get('sections', {}).items():
        for req_id, req in sec.get('requirements', {}).items():
            if req.get('thresholds'):
                for t in req['thresholds']:
                    print(f'  {t["threshold_id"]}: {t["metric"]} {t["operator"]} {t["value"]} {t["unit"]}')

2026-03-09 14:37:11,196 [INFO] execute_tool: Tool: traverse_compliance_path | inputs: ['entity_id', 'entity_type', 'regulation_id']
2026-03-09 14:37:11,402 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:37:11,463 [INFO] src.graph.connection: Neo4j connection closed.


Entity: {'loan_id': 'LOAN-0002', 'loan_type': None, 'amount': 900000, 'lvr': 95, 'interest_rate_pct': 5.11, 'borrower_id': 'BRW-0003', 'borrower_name': 'Katarina Wu', 'risk_rating': 'low', 'jurisdiction_id': 'JUR-AU-FED', 'jurisdiction_name': 'Federal', 'jurisdiction_aml_risk': 'low', 'collateral_value': 947368}
Jurisdiction: JUR-AU-FED

Regulations found: ['APG-223']
  APG-223-THR-001: risk_management_framework_review_frequency <= 3.0 years
  APG-223-THR-002: serviceability_policy_review_frequency <= 1.0 years
  APG-223-THR-003: interest_rate_serviceability_buffer >= 3.0 percent
  APG-223-THR-004: credit_card_revolving_debt_repayment_rate >= 3.0 percent
  APG-223-THR-005: buffer_and_floor_review_frequency <= 3.0 months
  APG-223-THR-006: non_salary_income_haircut >= 20.0 percent
  APG-223-THR-007: rental_income_haircut >= 20.0 percent
  APG-223-THR-008: LVR >= 90.0 percent


## Tool 3: retrieve_regulatory_chunks (FastMCP — vector search)

In [4]:
result = execute_tool('retrieve_regulatory_chunks', {
    'query_text': 'LVR limit high risk residential mortgage lending',
    'regulation_id': 'APG-223',
    'top_k': 3,
})

print(f'Query: {result["query"]}')
for chunk in result.get('chunks', []):
    print(f'\n[{chunk["chunk_id"]}] score={chunk["similarity_score"]}')
    print(f'  {chunk["text"][:200]}...')

2026-03-09 14:37:18,967 [INFO] execute_tool: Tool: retrieve_regulatory_chunks | inputs: ['query_text', 'regulation_id', 'top_k']
2026-03-09 14:37:19,895 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:37:19,929 [INFO] src.graph.connection: Neo4j connection closed.


Query: LVR limit high risk residential mortgage lending

[APG-223-CHUNK-0035] score=0.8285
  A prudent ADI would monitor exposures by LVR bands over time. Significant increases in high LVR lending would typically be a trigger for senior management to review risk targets and internal controls ...

[APG-223-CHUNK-0007] score=0.8246
  The overarching risk appetite statement required under CPS 220 would typically include an expression of the level of credit risk an ADI is willing to accept. Such a statement would be expected to adop...

[APG-223-CHUNK-0036] score=0.8228
  Sound credit practice would include recalculating LVRs at the time of any top-up loan and other formal loan increases during the life of the loan. Any subsequent refinancing, including any second mort...


## Tool 4: detect_graph_anomalies (FastMCP) — all 6 patterns

In [5]:
patterns = [
    'transaction_structuring',
    'high_lvr_loans',
    'high_risk_industry',
    'layered_ownership',
    'high_risk_jurisdiction',
    'guarantor_concentration',
]

for pattern in patterns:
    result = execute_tool('detect_graph_anomalies', {'pattern_name': pattern})
    count = result.get('finding_count', 0)
    sev   = result.get('severity', '?')
    ids   = result.get('entity_ids', [])[:5]
    status = '✓' if count > 0 else '○'
    print(f'{status} [{sev}] {pattern}: {count} findings  {ids}')

2026-03-09 14:37:21,733 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:37:21,938 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:37:21,983 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:37:21,984 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:37:22,159 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


✓ [HIGH] transaction_structuring: 3 findings  ['ACC-0610', 'ACC-0596', 'ACC-0603']


2026-03-09 14:37:22,221 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:37:22,221 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:37:22,392 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


✓ [HIGH] high_lvr_loans: 63 findings  ['LOAN-0002', 'LOAN-0448', 'LOAN-0033', 'LOAN-0077', 'LOAN-0159']


2026-03-09 14:37:22,470 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:37:22,470 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:37:22,645 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


...

✓ [MEDIUM] layered_ownership: 6 findings  ['BRW-0582', 'BRW-0587', 'BRW-0581', 'BRW-0582', 'BRW-0586']


2026-03-09 14:37:23,021 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:37:23,021 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 14:37:23,198 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


✓ [HIGH] high_risk_jurisdiction: 2 findings  ['BRW-0624', 'BRW-0627']


2026-03-09 14:37:23,272 [INFO] src.graph.connection: Neo4j connection closed.


○ [MEDIUM] guarantor_concentration: 0 findings  []


## Tool 5: persist_assessment + trace_evidence (FastMCP)

In [6]:
# Write a test assessment
result = execute_tool('persist_assessment', {
    'entity_id':     'LOAN-0002',
    'entity_type':   'LoanApplication',
    'regulation_id': 'APG-223',
    'verdict':       'REQUIRES_REVIEW',
    'confidence':    0.95,
    'findings': [{
        'finding_type': 'compliance_breach',
        'severity':     'HIGH',
        'description':  'LVR=95% breaches APG-223-THR-008 (LVR >= 90% requires senior review)',
        'pattern_name': 'high_lvr_loans',
    }],
    'reasoning_steps': [{
        'description':  'Retrieved LVR=95 from LoanApplication node via traverse_compliance_path',
        'cypher_used':  "MATCH (l:LoanApplication {loan_id: 'LOAN-0002'}) RETURN l.lvr",
        'section_ids':  ['APG-223-S4'],
        'chunk_ids':    [],
    }],
    'agent': 'notebook_test',
})
print('persist_assessment:', result)

assessment_id = result.get('assessment_id')

# Trace it back
if assessment_id:
    evidence = execute_tool('trace_evidence', {'assessment_id': assessment_id})
    print('\ntrace_evidence:')
    print(f'  Assessment: {evidence.get("assessment")}')
    print(f'  Findings:   {evidence.get("findings")}')
    print(f'  Steps:      {[s["description"] for s in evidence.get("reasoning_steps", [])]}')
    print(f'  Cited sections: {[s["section_id"] for s in evidence.get("cited_sections", [])]}')

2026-03-09 14:37:24,477 [INFO] execute_tool: Tool: persist_assessment | inputs: ['entity_id', 'entity_type', 'regulation_id', 'verdict', 'confidence', 'findings', 'reasoning_steps', 'agent']
2026-03-09 14:37:24,675 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 14:37:24,870 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 14:37:24,872 [INFO] execute_tool: Tool: trace_evidence | inputs: ['assessment_id']
2026-03-09 14:37:25,058 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


persist_assessment: {'assessment_id': 'ASSESS-LOAN-0002-APG-223-2026-03-09', 'finding_ids': ['FIND-ASSESS-LOAN-0002-APG-223-2026-03-09-000'], 'step_ids': ['STEP-ASSESS-LOAN-0002-APG-223-2026-03-09-000']}


2026-03-09 14:37:25,194 [WARNING] neo4j.notifications: Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `CITES_CHUNK` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=31, offset=171>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 171, 'line': 4, 'column': 31}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (a:Assessment {assessment_id: $id})-[:HAS_STEP]->(rs:ReasoningStep)\n        OPTIONAL MATCH (rs)-[:CITES_SECTION]->(s:Section)\n        OPTIONAL MATCH (rs)-[:CITES_CHUNK]->(c:Chunk)\n        RETURN rs.step_number   AS step_n


trace_evidence:
  Assessment: {'assessment_id': 'ASSESS-LOAN-0002-APG-223-2026-03-09', 'entity_id': 'LOAN-0002', 'entity_type': 'LoanApplication', 'regulation_id': 'APG-223', 'verdict': 'REQUIRES_REVIEW', 'confidence': 0.95, 'agent': 'notebook_test', 'created_at': '2026-03-09T03:37:24.488234+00:00'}
  Findings:   [{'finding_id': 'FIND-ASSESS-LOAN-0002-APG-223-2026-03-09-000', 'finding_type': 'compliance_breach', 'severity': 'HIGH', 'description': 'LVR=95% breaches APG-223-THR-008 (LVR >= 90% requires senior review)', 'pattern_name': 'high_lvr_loans'}]
  Steps:      ['Retrieved LVR=95 from LoanApplication node via traverse_compliance_path']
  Cited sections: ['APG-223-S4']


## Summary

All 7 tools tested. Ready to proceed to `313_compliance_agent.ipynb`.